In [2]:
import json
from pyspark.sql import SparkSession

# Inisialisasi SparkSession
spark = (
    SparkSession.builder.appName("LatihanModul2-6")
    .master("local[*]")
    .getOrCreate()
)

# ------------------------------------------------------------------
# DATASET MODUL 2, 3, 5 & 6: Data Transaksi (CSV), Produk (JSON), Ulasan (CSV)
# ------------------------------------------------------------------

# 1. transaksi.csv
csv_transaksi = """order_id,customer_id,produk_id,jumlah,harga_satuan,status,tanggal,kota
ORD01,CUST1,PROD_A,2,50000,SUCCESS,2026-01-10,Jakarta
ORD02,CUST2,PROD_B,1,150000,SUCCESS,2026-01-12,Surabaya
ORD03,CUST1,PROD_C,5,20000,FAILED,2026-01-15,Jakarta
ORD04,CUST3,PROD_A,3,50000,SUCCESS,2026-02-01,Bandung
ORD05,CUST2,PROD_C,2,20000,SUCCESS,2026-02-05,Surabaya
ORD06,CUST4,PROD_B,4,150000,SUCCESS,2026-02-10,Jakarta
ORD07,CUST5,PROD_A,1,50000,SUCCESS,2026-03-01,Medan
ORD08,CUST3,PROD_B,2,150000,FAILED,2026-03-05,Bandung"""

with open("transaksi.csv", "w") as f:
  f.write(csv_transaksi)

# 2. produk.json
json_produk = [
    {
        "produk_id": "PROD_A",
        "nama_produk": "Headphone Wireless",
        "kategori": "Elektronik",
        "manager": "Bapak Agus",
    },
    {
        "produk_id": "PROD_B",
        "nama_produk": "Kemeja Casual",
        "kategori": "Pakaian",
        "manager": "Ibu Nina",
    },
    {
        "produk_id": "PROD_C",
        "nama_produk": "Kopi Premium",
        "kategori": "Makanan",
        "manager": "Bapak Budi",
    },
]

with open("produk.json", "w") as f:
  json.dump(json_produk, f, indent=4)

# 3. ulasan.csv
csv_ulasan = """order_id,rating,komentar
ORD01,5,Sangat puas barang cepat sampai
ORD02,4,Bahan bagus sesuai ekspektasi
ORD04,5,Kualitas mantap
ORD05,2,Kemasan agak rusak
ORD06,4,Respon penjual cepat"""

with open("ulasan.csv", "w") as f:
  f.write(csv_ulasan)

# ------------------------------------------------------------------
# DATASET MODUL 4: Data Karyawan (Lengkap dengan Missing Values)
# ------------------------------------------------------------------
data_karyawan = [
    (101, "Budi Santoso", "IT", 8500000, 28),
    (102, "Siti Aminah", "Data", 11000000, 32),
    (103, "Andi Wijaya", None, 9000000, 45),
    (104, "Dewi Lestari", "IT", None, 24),
    (105, "Eko Prasetyo", "HR", 7000000, 38),
    (106, "Rina Putri", "Data", 12500000, 29),
    (107, "Fajar Nugraha", "Finance", None, 50),
    (108, "Gita Gutawa", None, 6500000, 26),
]
schema_karyawan = ["id_karyawan", "nama", "departemen", "gaji", "usia"]
df_karyawan = spark.createDataFrame(data_karyawan, schema_karyawan)
df_karyawan.write.mode("overwrite").csv("karyawan_raw.csv", header=True)

print("Dataset lokal berhasil dikonfigurasi!")

ModuleNotFoundError: No module named 'pyspark'

In [6]:
!hdfs dfs -put transaksi.csv /user/haris/latihanresponsi
!hdfs dfs -ls /user/haris/latihanresponsi

put: `/user/haris/latihanresponsi/transaksi.csv': File exists
Found 1 items
-rw-r--r--   1 haris supergroup        503 2026-10-01 12:21 /user/haris/latihanresponsi/transaksi.csv


In [7]:
!hdfs dfs -put produk.json /user/haris/latihanresponsi

In [8]:
!hdfs dfs -ls /user/haris/latihanresponsi

Found 2 items
-rw-r--r--   1 haris supergroup        448 2026-10-01 12:22 /user/haris/latihanresponsi/produk.json
-rw-r--r--   1 haris supergroup        503 2026-10-01 12:21 /user/haris/latihanresponsi/transaksi.csv


In [9]:
!hdfs dfs -cat /user/haris/latihanresponsi/transaksi.csv

order_id,customer_id,produk_id,jumlah,harga_satuan,status,tanggal,kota
ORD01,CUST1,PROD_A,2,50000,SUCCESS,2026-01-10,Jakarta
ORD02,CUST2,PROD_B,1,150000,SUCCESS,2026-01-12,Surabaya
ORD03,CUST1,PROD_C,5,20000,FAILED,2026-01-15,Jakarta
ORD04,CUST3,PROD_A,3,50000,SUCCESS,2026-02-01,Bandung
ORD05,CUST2,PROD_C,2,20000,SUCCESS,2026-02-05,Surabaya
ORD06,CUST4,PROD_B,4,150000,SUCCESS,2026-02-10,Jakarta
ORD07,CUST5,PROD_A,1,50000,SUCCESS,2026-03-01,Medan
ORD08,CUST3,PROD_B,2,150000,FAILED,2026-03-05,Bandung

In [13]:
df = spark.read.csv ("transaksi.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)

baris = df.count()
print("total baris adalah", baris )

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- produk_id: string (nullable = true)
 |-- jumlah: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- tanggal: date (nullable = true)
 |-- kota: string (nullable = true)

+--------+-----------+---------+------+------------+-------+----------+--------+
|order_id|customer_id|produk_id|jumlah|harga_satuan| status|   tanggal|    kota|
+--------+-----------+---------+------+------------+-------+----------+--------+
|   ORD01|      CUST1|   PROD_A|     2|       50000|SUCCESS|2026-01-10| Jakarta|
|   ORD02|      CUST2|   PROD_B|     1|      150000|SUCCESS|2026-01-12|Surabaya|
|   ORD03|      CUST1|   PROD_C|     5|       20000| FAILED|2026-01-15| Jakarta|
|   ORD04|      CUST3|   PROD_A|     3|       50000|SUCCESS|2026-02-01| Bandung|
|   ORD05|      CUST2|   PROD_C|     2|       20000|SUCCESS|2026-02-05|Surabaya|
+--------+-----------+-----

In [17]:
df_new = df.select("order_id", "kota", "jumlah", "harga_satuan")
df_new.show(5)

+--------+--------+------+------------+
|order_id|    kota|jumlah|harga_satuan|
+--------+--------+------+------------+
|   ORD01| Jakarta|     2|       50000|
|   ORD02|Surabaya|     1|      150000|
|   ORD03| Jakarta|     5|       20000|
|   ORD04| Bandung|     3|       50000|
|   ORD05|Surabaya|     2|       20000|
+--------+--------+------+------------+
only showing top 5 rows



In [26]:
from pyspark.sql.functions import col
df_total_belanja = df.withColumn("total", col("harga_satuan") * col("jumlah"))
df_total_belanja.show(5)

+--------+-----------+---------+------+------------+-------+----------+--------+------+
|order_id|customer_id|produk_id|jumlah|harga_satuan| status|   tanggal|    kota| total|
+--------+-----------+---------+------+------------+-------+----------+--------+------+
|   ORD01|      CUST1|   PROD_A|     2|       50000|SUCCESS|2026-01-10| Jakarta|100000|
|   ORD02|      CUST2|   PROD_B|     1|      150000|SUCCESS|2026-01-12|Surabaya|150000|
|   ORD03|      CUST1|   PROD_C|     5|       20000| FAILED|2026-01-15| Jakarta|100000|
|   ORD04|      CUST3|   PROD_A|     3|       50000|SUCCESS|2026-02-01| Bandung|150000|
|   ORD05|      CUST2|   PROD_C|     2|       20000|SUCCESS|2026-02-05|Surabaya| 40000|
+--------+-----------+---------+------+------------+-------+----------+--------+------+
only showing top 5 rows



In [25]:
df.describe("jumlah", "harga_satuan").show(5)

+-------+------------------+-----------------+
|summary|            jumlah|     harga_satuan|
+-------+------------------+-----------------+
|  count|                 8|                8|
|   mean|               2.5|          80000.0|
| stddev|1.4142135623730951|59281.41120356123|
|    min|                 1|            20000|
|    max|                 5|           150000|
+-------+------------------+-----------------+



In [42]:
from pyspark.sql.functions import sum as spark_sum, count, avg, col

# 1. Baca file CSV dengan header dan inferSchema agar tipe data numerik terbaca dengan benar
df_transaksi = spark.read.csv("transaksi.csv", header=True, inferSchema=True)

# 2. Tambahkan kolom turunan total_belanja (jumlah * harga_satuan)
df_transaksi = df_transaksi.withColumn("total_belanja", col("jumlah") * col("harga_satuan"))

# 3. Filter status "SUCCESS" dan kota "Jakarta" atau "Surabaya"
df_filtered = df_transaksi.filter(
    (col("status") == "SUCCESS") & col("kota").isin("Jakarta", "Surabaya")
)

# 4. Agregasi data per kota dan urutkan dari total pendapatan tertinggi
new = df_filtered.groupBy("kota").agg(
    spark_sum("total_belanja").alias("total_pendapatan"),
    avg("jumlah").alias("rata_rata_jumlah"),
    count("order_id").alias("total_transaksi")
).orderBy(col("total_pendapatan").desc())

new.show()

+--------+----------------+----------------+---------------+
|    kota|total_pendapatan|rata_rata_jumlah|total_transaksi|
+--------+----------------+----------------+---------------+
| Jakarta|          700000|             3.0|              2|
|Surabaya|          190000|             1.5|              2|
+--------+----------------+----------------+---------------+



In [1]:
spark.stop()

NameError: name 'spark' is not defined